# Создание файлов для ручной валидации

In [1]:
import numpy as np
import pandas as pd

from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.worksheet.datavalidation import DataValidation

In [3]:
post_base = pd.read_csv('/Users/ekaterinamalkova/Desktop/vkr_analysis/data/posts_for_validation.csv')
comment_base = pd.read_csv('/Users/ekaterinamalkova/Desktop/vkr_analysis/data/comments_for_validation.csv')

In [ ]:
post_core_parts = []

for _, group in post_base.groupby('narrative_class'):
    post_core_parts.append(group.sample(n=min(120,len(group)), random_state=42))
post_core = pd.concat(post_core_parts)

post_hard_candidates = post_base[(post_base['narrative_class'] == 'Смешанный: престижность и доступность')
    | (post_base['prestige_marker_count'].between(1, 2)
        & post_base['access_marker_count'].between(1, 2))]

post_hard = post_hard_candidates.sample(n=min(120, len(post_hard_candidates)), random_state=42)

engagement_score = (post_base['likes_p1000_views'].rank(pct=True)
                    + post_base['comments_p1000_views'].rank(pct=True)
                    + post_base['reposts_p1000_views'].rank(pct=True))

post_high_impact = post_base.loc[engagement_score.nlargest(120).index]

post_sample = (pd.concat([post_core, post_hard, post_high_impact])
                .drop_duplicates(['post_id', 'source_nickname'])
                .sample(frac=1, random_state=42)
                .head(600)
                .reset_index(drop=True))

if len(post_sample) <600:
    post_sample_key = set(zip(post_sample['post_id'], post_sample['source_nickname']))
    post_remaining = post_base[~post_base.apply(lambda row: (row['post_id'], row['source_nickname']) in post_sample_key, axis=1)]
    post_extra = post_remaining.sample(n=min(600- len(post_sample), len(post_remaining)), random_state=42)
    post_sample = pd.concat([post_sample, post_extra], ignore_index=True).head(600)

post_sample.insert(0, 'validation_id', [f'P{i:04d}' for i in range(1,len(post_sample) +1)])
post_sample['manual_narrative_label'] = ''
post_sample['manual_prestige_subtype'] = ''
post_sample['manual_access_subtype'] = ''
post_sample['manual_sentiment_label'] = ''
post_sample['manual_relevance'] = ''
post_sample['annotation_confidence_1_3'] = ''
post_sample['manual_notes'] = ''

post_columns = ['validation_id', 'post_id', 'date', 'year', 'month', 'source_nickname',
                'narrative_class', 'prestige_marker_count', 'access_marker_count',
                'prestige_density', 'access_density', 'sentiment_label', 'sentiment_score',
                'likes', 'comments_count', 'reposts', 'views',
                'likes_p1000_views', 'comments_p1000_views', 'reposts_p1000_views',
                'text_for_annotation',
                'manual_narrative_label', 'manual_prestige_subtype', 'manual_access_subtype',
                'manual_sentiment_label', 'manual_relevance', 'annotation_confidence_1_3', 'manual_notes']
post_columns = [col for col in post_columns if col in post_sample.columns]
post_sample = post_sample[post_columns]
post_sample.to_csv('post_sample.csv', index=False)

post_sample['narrative_class'].value_counts()

In [ ]:
comment_base = comment_base.drop(columns=['Unnamed: 0'], errors='ignore')

comment_sample_parts = []

for _, group in comment_base.groupby('comments_validation'):
    comment_sample_parts.append(
        group.sample(n=min(80, len(group)),random_state=42))

comment_sample = (pd.concat(comment_sample_parts)
                    .drop_duplicates(['post_id', 'source_nickname'])
                    .sample(frac=1, random_state=42)
                    .head(400)
                    .reset_index(drop=True))

comment_sample.insert(0,'validation_id',[f'C{i:04d}' for i in range(1,len(comment_sample) +1)])

comment_sample['manual_comment_reaction_type'] = ''
comment_sample['manual_contains_question'] = ''
comment_sample['manual_contains_barrier_discussion'] = ''
comment_sample['manual_comment_sentiment'] = ''
comment_sample['manual_contest_or_mechanical'] = ''
comment_sample['annotation_confidence_1_3'] = ''
comment_sample['manual_notes'] = ''

comment_columns = ['validation_id', 'post_id', 'date', 'year', 'month', 'source_nickname',
                    'comments_validation', 'narrative_class', 'comments_count',
                    'contest_like_flag', 'comment_meaningful_part', 'comment_question_part',
                    'comment_short_part', 'comment_duplicate_part', 'comment_ritualistic_part',
                    'meaningful_comm_clean', 'comment_access_barrier_mentions',
                    'text_for_annotation', 'comments_for_annotation', 'meaningful_comments_clean',
                    'manual_comment_reaction_type', 'manual_contains_question',
                    'manual_contains_barrier_discussion', 'manual_comment_sentiment',
                    'manual_contest_or_mechanical', 'annotation_confidence_1_3', 'manual_notes']

comment_columns = [col for col in comment_columns if col in comment_sample.columns]
comment_sample = comment_sample[comment_columns]

comment_sample.to_csv('comment_sample.csv', index=False)

comment_sample['comments_validation'].value_counts()

In [5]:
excel_files = [(post_sample, 'post_val.xlsx', 'posts_validation'),
               (comment_sample, 'comms_val.xlsx', 'comments_validation')]


for data, path, sheet_name in excel_files:
    with pd.ExcelWriter(path, engine='openpyxl') as writer:
        data.to_excel(writer, index=False, sheet_name=sheet_name)

    wb = load_workbook(path)
    ws = wb[sheet_name]

    ws.freeze_panes = 'B2'
    ws.auto_filter.ref = ws.dimensions

    header_fill = PatternFill('solid', fgColor='1F4E78')
    manual_fill = PatternFill('solid', fgColor='FFF2CC')
    text_fill = PatternFill('solid', fgColor='E2F0D9')
    header_font = Font(color='FFFFFF', bold=True)

    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(
            horizontal='center',
            vertical='center',
            wrap_text=True)

    col_index = {cell.value: i for i, cell in enumerate(ws[1], start=1)}

    manual_columns = [c for c in data.columns if c.startswith('manual_') or c == 'annotation_confidence_1_3']

    text_columns = ['text_for_annotation',
                    'comments_for_annotation',
                    'meaningful_comments_clean',
                    'manual_notes']

    for col_name in manual_columns:
        col_num = col_index.get(col_name)
        if col_num is not None:
            for row in range(2, ws.max_row + 1):
                ws.cell(row=row, column=col_num).fill = manual_fill

    for col_name in text_columns:
        col_num = col_index.get(col_name)
        if col_num is not None:
            for row in range(2, ws.max_row + 1):
                ws.cell(row=row, column=col_num).alignment = Alignment(vertical='top',
                                                                       wrap_text=True)
                if col_name != 'manual_notes':
                    ws.cell(row=row, column=col_num).fill = text_fill

    dropdowns = {'manual_narrative_label': 'prestige,access,mixed,other,irrelevant',
                'manual_prestige_subtype': 'selectivity,elite_status,achievement,scientific_productivity,internationalization,career_capital,none',
                'manual_access_subtype': 'financial,procedural,informational,infrastructure_support,none',
                'manual_sentiment_label': 'positive,neutral,negative,mixed',
                'manual_relevance': 'relevant,partly_relevant,irrelevant',
                'manual_comment_reaction_type': 'substantive,question,critique,gratitude,ritual,contest,spam,unclear',
                'manual_contains_question': 'yes,no',
                'manual_contains_barrier_discussion': 'yes,no',
                'manual_comment_sentiment': 'positive,neutral,negative,mixed',
                'manual_contest_or_mechanical': 'yes,no',
                'annotation_confidence_1_3': '1,2,3',}

    for col_name, options in dropdowns.items():
        col_num = col_index.get(col_name)

        if col_num is not None:
            col_letter = ws.cell(row=1, column=col_num).column_letter

            validation = DataValidation(type='list',formula1=f'"{options}"', allow_blank=True)
            ws.add_data_validation(validation)
            validation.add(f'{col_letter}2:{col_letter}{ws.max_row}')

    for column_cells in ws.columns:
        header = column_cells[0].value

        if header in ['text_for_annotation',
                      'comments_for_annotation',
                      'meaningful_comments_clean']:
            width = 70
        elif header == 'manual_notes':
            width = 38
        elif header in manual_columns:
            width = 28
        else:
            width = 16

        ws.column_dimensions[column_cells[0].column_letter].width = width

    ws.row_dimensions[1].height = 32

    for row in range(2, min(ws.max_row, 120) + 1):
        ws.row_dimensions[row].height = 72

    wb.save(path)

## Валидация

### Чтение файлов с ручной разметкой

In [6]:
posts_val = pd.read_csv('/Users/ekaterinamalkova/Desktop/vkr_analysis/data/validation_data/post_val.csv',
    sep=';', skiprows=1, engine="python", dtype=str)
comms_val = pd.read_csv('/Users/ekaterinamalkova/Desktop/vkr_analysis/data/validation_data/comms_val.csv',
    sep=';', skiprows=1, engine="python", dtype=str)

### Переименование признаков

In [7]:
post_needed = ['manual_narrative_label',
               'manual_prestige_subtype', 'manual_access_subtype',
               'manual_sentiment_label', 'manual_relevance',
               'annotation_confidence_1_3']
comm_needed = ['manual_comment_reaction_type', 'manual_contains_question',
               'manual_contains_barrier_discussion', 'manual_comment_sentiment',
               'manual_contest_or_mechanical', 'annotation_confidence_1_3']

auto_names = {'narrative_class': 'auto_narrative_label',
              'sentiment_label': 'auto_sentiment_label',
              'comments_validation': 'auto_comment_reaction_type',
              'comment_question_part': 'auto_contains_question',
              'comment_access_barrier_mentions': 'auto_contains_barrier_discussion',
              'contest_like_flag': 'auto_contest_or_mechanical'}

narrative_values = {'Престижность': 'prestige',
                    'Доступность': 'access',
                    'Смешанный: престижность и доступность': 'mixed',
                    'Другое': 'other'}

sentiment_values = {'Позитивная': 'positive',
                    'Нейтральная': 'neutral',
                    'Негативная': 'negative'}

comment_type_values = {'question_heavy': 'question',
                       'contest_or_mechanical': 'contest',
                       'meaningful_heavy': 'substantive',
                       'short_or_ritualistic': 'ritual',
                       'ordinary': 'substantive'}


block_names = {'post_narrative': 'Нарратив постов',
               'post_sentiment': 'Тональность постов',
               'comment_reaction_type': 'Тип реакции комментариев',
               'comment_contains_question': 'Вопросы в комментариях',
               'comment_contains_barrier_discussion': 'Барьеры доступности',
               'comment_contest_or_mechanical': 'Конкурсная/механическая активность'}

metric_column_names = {'block': 'Проверяем',
                       'n': 'N',
                       'accuracy': 'accuracy',
                       'macro_f1': 'macro_f1',
                       'precision_yes': 'precision_yes',
                       'recall_yes': 'recall_yes',
                       'f1_yes': 'f1_yes',
                       'support_yes': 'support_yes, N',
                       'support_no': 'support_no, N'}

confusion_column_names = {'block': 'Проверяем', 
                          'manual': 'Ручная метка',
                          'auto': 'Автометка',
                          'n': 'N'}

### Вспомогательные функции

In [8]:
def multiclass_metrics(y_true, y_pred,labels):
    mask = y_true.notna() & y_pred.notna()
    y_true = y_true[mask].astype(str)
    y_pred = y_pred[mask].astype(str)
    present = [label for label in labels if label in set(y_true) or label in set(y_pred)]

    matrix = pd.DataFrame([[int(((y_true == true_label) & (y_pred == pred_label)).sum()) for pred_label in present] for true_label in present],
                            index=present,
                            columns=present)
    f1_scores = []
    for label in present:
        tp = int(((y_true == label) & (y_pred == label)).sum())
        fp = int(((y_true != label) & (y_pred == label)).sum())
        fn = int(((y_true ==label) & (y_pred !=label)).sum())
        precision = tp / (tp + fp) if (tp +fp) else 0.0
        recall = tp / (tp + fn) if (tp + fn) else 0.0
        f1_scores.append(2 * precision * recall / (precision + recall) if (precision + recall) else 0.0)

    return {'n': int(len(y_true)),
            'accuracy': float((y_true ==y_pred).mean()) if len(y_true) else np.nan,
            'f1score': float(np.mean(f1_scores)) if f1_scores else np.nan,}, matrix

In [9]:
def binary_metrics(y_true_yes, y_pred_yes):
    mask = y_true_yes.notna()& y_pred_yes.notna()
    y_true = y_true_yes[mask].astype(bool)
    y_pred = y_pred_yes[mask].astype(bool)

    tp = int((y_true &y_pred).sum())
    fp = int((~y_true &y_pred).sum())
    fn = int((y_true & ~y_pred).sum())
    precision = tp / (tp + fp) if (tp +fp) else 0.0
    recall = tp /(tp + fn) if (tp + fn) else 0.0
    f1 = 2 * precision * recall /(precision +recall) if (precision +recall) else 0.0

    return {'n': int(len(y_true)),
            'accuracy': float((y_true == y_pred).mean()) if len(y_true) else np.nan,
            'precision_yes':precision,
            'recall_yes': recall,
            'f1_yes': f1,
            'support_yes':int(y_true.sum()),
            'support_no': int((~y_true).sum())}


### Работа с признаками

In [10]:
posts_val[auto_names['narrative_class']] = posts_val['narrative_class'].map(narrative_values)
posts_val[auto_names['sentiment_label']] = posts_val['sentiment_label'].map(sentiment_values)

comms_val[auto_names['comments_validation']] = comms_val['comments_validation'].map(comment_type_values)
comms_val[auto_names['comment_question_part']] = pd.to_numeric(
    comms_val['comment_question_part'].astype(str).str.replace(',', '.', regex=False),
    errors='coerce').fillna(0)>0
comms_val[auto_names['comment_access_barrier_mentions']] = pd.to_numeric(
    comms_val['comment_access_barrier_mentions'].astype(str).str.replace(',', '.', regex=False),
    errors='coerce').fillna(0)>0
comms_val[auto_names['contest_like_flag']] = comms_val['contest_like_flag'].astype(str).str.strip().str.lower().isin(
    {'true','истина','1', 'yes','да'})

manual_question = comms_val['manual_contains_question'].astype('string').str.strip().str.lower().map({'yes': True, 'no': False})
manual_barrier = comms_val['manual_contains_barrier_discussion'].astype('string').str.strip().str.lower().map({'yes': True, 'no': False})
manual_mechanical = comms_val['manual_contest_or_mechanical'].astype('string').str.strip().str.lower().map({'yes': True, 'no': False})

post_narrative_m, post_narrative_cm = multiclass_metrics(posts_val['manual_narrative_label'],
                                                         posts_val[auto_names['narrative_class']],
                                                         ['prestige', 'access', 'mixed', 'other', 'irrelevant'])
post_sentiment_m, post_sentiment_cm = multiclass_metrics(posts_val['manual_sentiment_label'],
                                                         posts_val[auto_names['sentiment_label']],
                                                         ['positive', 'neutral', 'negative', 'mixed'])
comment_type_m, comment_type_cm = multiclass_metrics(comms_val['manual_comment_reaction_type'],
                                                     comms_val[auto_names['comments_validation']],
                                                     ['substantive', 'question', 'critique', 'gratitude', 'ritual', 'contest', 'spam', 'unclear'])

question_m = binary_metrics(manual_question, comms_val[auto_names['comment_question_part']])
barrier_m = binary_metrics(manual_barrier, comms_val[auto_names['comment_access_barrier_mentions']])
mechanical_m = binary_metrics(manual_mechanical, comms_val[auto_names['contest_like_flag']])

### Рассчет метрик

Объем и полнота выполненной разметки – 600 постов и 400 постов со слоем комментариев

In [11]:
volume = pd.DataFrame([{'Выборка': 'Посты',
                        'Строк': len(posts_val),
                        'Размечено': int(posts_val[post_needed].notna().all(axis=1).sum())},
                       {'Выборка': 'Комментарии',
                        'Строк': len(comms_val),
                        'Размечено': int(comms_val[comm_needed].notna().all(axis=1).sum())}])
volume

,Выборка,Строк,Размечено
0,Посты,600,600
1,Комментарии,400,400


Сравнение ручных и авто -метками для нарратива, тональности и реакции

In [12]:
multi_compare = pd.DataFrame([{'block': 'post_narrative', **post_narrative_m},
                              {'block': 'post_sentiment', **post_sentiment_m},
                              {'block': 'comment_reaction_type', **comment_type_m}])
multi_compare[['accuracy', 'f1score']] = (multi_compare[['accuracy', 'f1score']]).round(2)
multi_compare = multi_compare.replace({'block': block_names}).rename(columns=metric_column_names)
multi_compare

,Проверяем,N,accuracy,f1score
0,Нарратив постов,600,0.34,0.25
1,Тональность постов,600,0.58,0.34
2,Тип реакции комментариев,400,0.37,0.21


Сравнение метрик по бинарным признакам слоя комментариев

In [13]:
binary_compare = pd.DataFrame([{'block': 'comment_contains_question', **question_m},
                               {'block': 'comment_contains_barrier_discussion', **barrier_m},
                               {'block': 'comment_contest_or_mechanical', **mechanical_m},])
binary_compare[['accuracy', 'precision_yes', 'recall_yes', 'f1_yes']] = (binary_compare[
    ['accuracy', 'precision_yes', 'recall_yes', 'f1_yes']]).round(2)
binary_compare = binary_compare.replace({'block': block_names}).rename(columns=metric_column_names)
binary_compare

,Проверяем,N,accuracy,precision_yes,recall_yes,f1_yes,"support_yes, N","support_no, N"
0,Вопросы в комментариях,400,0.76,0.44,1.00,0.61,75,325
1,Барьеры доступности,400,0.91,0.08,1.00,0.15,3,397
2,Конкурсная/механическая активность,400,0.76,0.25,0.37,0.30,54,346


Посмотрим на расхождения в ручной и авто -метках

In [14]:
diff_sources = {'post_narrative': post_narrative_cm,
                'post_sentiment': post_sentiment_cm,
                'comment_reaction_type': comment_type_cm}

main_diff = (pd.concat(diff_sources, names=['block'])
               .rename_axis(index=['block', 'manual'], columns='auto')
               .stack()
               .rename('n')
               .reset_index()
               .query('manual !=auto and n >0')
               .sort_values(['block', 'n'], ascending=[True, False])
               .groupby('block')
               .head(5)
               .replace({'block': block_names})
               .rename(columns=confusion_column_names))
main_diff

,Проверяем,Ручная метка,Автометка,N
58,Тип реакции комментариев,gratitude,substantive,52.0
71,Тип реакции комментариев,ritual,contest,28.0
66,Тип реакции комментариев,ritual,substantive,23.0
39,Тип реакции комментариев,substantive,contest,21.0
35,Тип реакции комментариев,substantive,question,20.0
17,Нарратив постов,other,mixed,113.0
15,Нарратив постов,other,prestige,71.0
16,Нарратив постов,other,access,54.0
2,Нарратив постов,prestige,mixed,44.0
7,Нарратив постов,access,mixed,41.0


## Выводы этапа  
  1. Автоматические правила совпадают с ручной разметкой довольно слабо, поэтому они позиционируются как weak labelling.
  2. Бинарный признак вопросов в комментариях – авто нашел все ручные разметки, но нашел и много лишних; признак обсуждения барьеров доступности сложно оценивать по метрикам, тк там сильный дисбаланс классов (всего 3 из 400 yes); по конкурсной/механической активности тоже есть пропуски реальных подходящих комментариев, а также лишние срабатывания.
  3. Автоматические правила слишком часто видят нарратив там, где ручной разметкой публикация отнесена к "другое". По тональности авто часто переоценивает позитивность постов, когда ручная разметка относит публикацию к нейтральной. Также разделение типа комментария (конкурсный, ритуальный и тд) довольно запутанное. Важно отметить, что нельзя исключать излишнюю строгость ручной разметки без ее кросс-валидации. Всвязи с этим возможен более грубый подход к оцениванию материала, а также возникновение ошибок и расхождений.